## ConversationBufferMemory

In [1]:
import os

from dotenv import load_dotenv, find_dotenv
_ = load_dotenv(find_dotenv()) # read local .env file

import warnings
warnings.filterwarnings('ignore')

In [2]:
# account for deprecation of LLM model
import datetime
# Get the current date
current_date = datetime.datetime.now().date()

# Define the date after which the model should be set to "gpt-3.5-turbo"
target_date = datetime.date(2024, 6, 12)

# Set the model variable based on the current date
if current_date > target_date:
    llm_model = "gpt-3.5-turbo"
else:
    llm_model = "gpt-3.5-turbo-0301"

In [ ]:
#!pip install langchain_community

from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, AIMessage


#Deprecated
#from langchain.chat_models import ChatOpenAI   
#from langchain.chains import ConversationChain
#from langchain.memory import ConversationBufferMemory



from langchain_openai import ChatOpenAI


llm = ChatOpenAI(
    model=llm_model,
    temperature=0.0,
    openai_api_key=os.environ.get("OPENAI_API_KEY")
)

messages = []

def chat(user_input: str):
    messages.append(HumanMessage(content=user_input))

    response = llm.invoke(messages)

    messages.append(AIMessage(content=response.content))

    return response.content


print(chat("Hi, my name is Andrew"))


Hello Andrew, nice to meet you! How can I assist you today?


In [20]:
print(chat("What is 1+1?"))

1+1 equals 2.


In [21]:
print(chat("What's my name?"))

Your name is Andrew.


In [24]:
print(messages)

[HumanMessage(content='Hi, my name is Andrew', additional_kwargs={}, response_metadata={}), AIMessage(content='Hello Andrew, nice to meet you! How can I assist you today?', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]), HumanMessage(content='What is 1+1?', additional_kwargs={}, response_metadata={}), AIMessage(content='1+1 equals 2.', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]), HumanMessage(content="What's my name?", additional_kwargs={}, response_metadata={}), AIMessage(content='Your name is Andrew.', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[])]


In [25]:
for m in messages:
    print(f"{m.type}: {m.content}")


human: Hi, my name is Andrew
ai: Hello Andrew, nice to meet you! How can I assist you today?
human: What is 1+1?
ai: 1+1 equals 2.
human: What's my name?
ai: Your name is Andrew.


In [26]:
from langchain_core.chat_history import InMemoryChatMessageHistory
from langchain_core.messages import get_buffer_string

memory = InMemoryChatMessageHistory()

memory.add_user_message("Hi")
memory.add_ai_message("What's up")

print(get_buffer_string(memory.messages))   # equivalent to memory.buffer
print({"history": get_buffer_string(memory.messages)})  # equivalent to load_memory_variables({})

memory.add_user_message("Not much, just hanging")
memory.add_ai_message("Cool")

print(get_buffer_string(memory.messages))
print({"history": get_buffer_string(memory.messages)})


Human: Hi
AI: What's up
{'history': "Human: Hi\nAI: What's up"}
Human: Hi
AI: What's up
Human: Not much, just hanging
AI: Cool
{'history': "Human: Hi\nAI: What's up\nHuman: Not much, just hanging\nAI: Cool"}


## ConversationBufferWindowMemory

In [8]:
from langchain_openai import ChatOpenAI

In [ ]:
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_core.chat_history import InMemoryChatMessageHistory
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.messages import trim_messages

In [13]:
llm_model = "gpt-3.5-turbo"

In [14]:
llm = ChatOpenAI(temperature=0.0, model=llm_model)

In [15]:
# Store for session histories (swap for a DB-backed store in production)
store = {}

In [16]:
def get_session_history(session_id: str) -> InMemoryChatMessageHistory:
    if session_id not in store:
        store[session_id] = InMemoryChatMessageHistory()
    return store[session_id]

In [17]:
# Windowing: keep only the last k exchanges (mimics k=1 window)
trimmer = trim_messages(
    max_tokens=1,       # counts "messages" here via token_counter below
    strategy="last",
    token_counter=lambda msgs: len(msgs),  # counts messages, not tokens
    include_system=True,
)
# For k=1 (last 1 human+ai pair = 2 messages), set max_tokens=2

In [18]:
prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant."),
    MessagesPlaceholder(variable_name="history"),
    ("human", "{input}"),
])

In [19]:
chain = prompt | llm

In [20]:
conversation = RunnableWithMessageHistory(
    chain,
    get_session_history,
    input_messages_key="input",
    history_messages_key="history",
)